In [ ]:
'''
Function: ner_few_shot
Description:
 - Performs few-shot Named Entity Recognition (NER) using the GPT-4o model.
 - Takes instructions, examples, input text, model name, and temperature value as parameters.
 - Sends the instructions and the text to the OpenAI chat API.
 - Receives and cleans the model's response.
 - Attempts to parse the response as JSON.
 - Returns the parsed output if valid, otherwise returns None.
'''

from openai import OpenAI
import json

def ner_few_shot(instruction, example, paragraph, model, temperature):
    client = OpenAI()
    user_query = '''
        EXAMPLES:
        {example}
        
        TEXT:
        {paragraph}
    '''
    
    response = client.chat.completions.create(
      model = model,
      temperature = temperature,
      messages = [
        {'role': 'system', 'content': instruction},
        {'role': 'user', 'content': user_query.format(example=example, text=paragraph)}
      ]
    )

    content = response.choices[0].message.content
    cleaned_content = content.strip('```python\n').strip('```')
    
    try:
        output = json.loads(cleaned_content)
        return output
    except json.JSONDecodeError as e:
        return None
    

In [ ]:
# read evaluation data from text file and store paragraphs and terms in different list

import json

def process_input_data(text_file):
    # read input data from text file
    with open(text_file, 'r', encoding='utf-8') as file:
        list_ = file.read().splitlines()

    # store paragraphs and annotations in different lists
    paragraph = []
    gold_ent_str = []

    for item in list_:
        if list_.index(item) == 0 or list_.index(item) % 2 == 0:
            paragraph.append(item)
        else:
            gold_ent_str.append(item)

    # convert annotations (in string format) to nested object
    gold_entity = []

    for item in gold_ent_str:
        try:
            json_obj = json.loads(item)              # Convert to dictionary
            gold_entity.append(json_obj)  # Add to list of dictionaries
        except json.JSONDecodeError as e:
            print(f'Error decoding JSON for item: {item}\nError: {e}')

    return paragraph, gold_entity
    


def annotate(instruction, example, paragraph, label, gold_entity, model, temp, dir_path):
    # predict entities from each paragraph for every instruction
    pred_entity = []
    for para in paragraph:
        pred_ent_in_para = dict()

        for inst, exmp in zip(instruction, example):
            response = ner_few_shot(inst, exmp, para, model, temp)
            if response:
                llm_label = list(response.keys())
                llm_ent = list(response.values())
                pred_ent_in_para[llm_label[0]] = llm_ent[0]
    #         else:
    #             print("ERROR::", response)

        # check for missing labels in predicted entities
        # all labels should be there even if they do not have any entities 
        if len(pred_ent_in_para) < len(label):
            print('Label missing in predicted data.')
            revised_data = dict()

            for l in label:
                if l not in pred_ent_in_para:
                    pred_ent_in_para[l] = []
                    print(f'Label -- {l} -- added to predicted data.')

            # organize the annotations' labels according to label's order
            for l in label:
                revised_data[l] = pred_ent_in_para[l]

            pred_ent_in_para = revised_data

        pred_entity.append(pred_ent_in_para)

    # save label, paragraph, gold_entity, pred_entity variables
    with open(f'{dir_path}\\evaluation_variable.py', 'w', encoding='utf-8') as file:
        file.write('label = ' + repr(label) + '\n')
        file.write('paragraph = ' + repr(paragraph) + '\n')
        file.write('gold_entity = ' + repr(gold_entity) + '\n')
        file.write('pred_entity = ' + repr(pred_entity) + '\n')

    print(f'Downloaded evaluation_variable.py => {dir_path}')

    return pred_entity


In [ ]:
from chatgpt_prompt import instructions, examples_20

labels = [
    'chemical',
    'material',
    'structure',
    'property',
    'application',
    'process',
    'equipment',
    'measurement',
    'abbreviation'
]

paragraphs, gs_ents = process_input_data(text_file='')
pred_ents = annotate(
                instructions=instructions,
                examples=examples_20,
                paragraphs=paragraphs,
                labels=labels,
                model='gpt-4o',
                temperature=0.2,
                directory_path='output/few-shot'
            )

eval_distinct_ents(
    labels=labels,
    paragraphs=paragraphs,
    gold_standard_entities=gs_ents,
    predicted_entities=pred_ents,
    directory_path='output/few-shot'
)

eval_all_ents(
    label=labels,
    para=paragraphs,
    gs_ents=gs_ents,
    pred_ents=pred_ents,
    dir_path='output/few-shot'
)